# ML Intraday V3 — Pipeline Runner
Runs and inspects the **ml_intraday_v3** pipeline stages.

**Assumptions**
- Run from the repo root (contains `ml_intraday_v3/`).
- Virtual env is activated and deps installed.
- `ml_intraday_v3/configs/data.yaml` points to your raw data.


## 0) Parameters

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import uuid
import os, json, subprocess, sys
import importlib.util
import pandas as pd

RUN_ID = os.environ.get("MLV3_RUN_ID")
if not RUN_ID:
    RUN_ID = "run_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
os.environ["MLV3_RUN_ID"] = RUN_ID
REPO_ROOT = Path(".").resolve()
if not (REPO_ROOT / "ml_intraday_v3").exists():
    for parent in REPO_ROOT.parents:
        if (parent / "ml_intraday_v3").exists():
            REPO_ROOT = parent
            break
    else:
        raise RuntimeError("Repo root not found. Run this notebook from the repo root.")

PYTHON = sys.executable

CLI_MODULE = "ml_intraday_v3.cli"

CONFIG_DIR = REPO_ROOT / "ml_intraday_v3" / "configs"
DATA_YAML = CONFIG_DIR / "data.yaml"
FEATURES_YAML = CONFIG_DIR / "features.yaml"
LABELING_YAML = CONFIG_DIR / "labeling.yaml"
VALIDATION_YAML = CONFIG_DIR / "validation.yaml"
TRAINING_YAML = CONFIG_DIR / "training.yaml"
BACKTEST_YAML = CONFIG_DIR / "backtest.yaml"
EXPERIMENT_GRID_YAML = CONFIG_DIR / "experiment_grid.yaml"
WALKFORWARD_YAML = CONFIG_DIR / "walkforward.yaml"

EXECUTION_SPEC_YAML = CONFIG_DIR / "execution_spec.yaml"
RISK_YAML = CONFIG_DIR / "risk.yaml"

RUN_DIR = REPO_ROOT / "runs" / RUN_ID
BAR_SIZES = ["1m", "5m"]
SEED = 42
CV_KIND = "purged_kfold"

HAS_PYARROW = importlib.util.find_spec("pyarrow") is not None

print("Repo root:", REPO_ROOT)
print("Run ID:", RUN_ID)
print("Run dir:", RUN_DIR)
print("Python:", PYTHON)
print("pyarrow available:", HAS_PYARROW)


Repo root: /Users/eshaanganguly/Documents/projects/algos 3 topstep
Run ID: baseline_v3_001
Run dir: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001
Python: /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python
pyarrow available: True


## 1) Helpers

In [2]:
from pathlib import Path
import sys, os

def run_cmd(cmd, check=True):
    # cmd: list[str]
    cmd = list(cmd)
    if cmd and cmd[0] == "python":
        cmd[0] = sys.executable
    elif cmd and cmd[0] == "pytest":
        cmd = [sys.executable, "-m", "pytest"] + cmd[1:]

    env = os.environ.copy()
    repo_root = str(REPO_ROOT)
    existing = env.get("PYTHONPATH", "")
    if repo_root not in existing.split(os.pathsep):
        env["PYTHONPATH"] = repo_root + (os.pathsep + existing if existing else "")

    print(">>", " ".join(cmd))
    p = subprocess.run(cmd, text=True, capture_output=True, cwd=repo_root, env=env)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}")
    return p

def show_dir(path: Path, max_lines=200):
    path = Path(path)
    if not path.exists():
        print("Missing:", path)
        return
    lines = []
    for p in sorted(path.rglob("*")):
        if p.is_file():
            lines.append(str(p.relative_to(path)))
    print("\n".join(lines[:max_lines]))
    if len(lines) > max_lines:
        print(f"... ({len(lines) - max_lines} more)")

def read_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)

def read_parquet(path: Path, columns=None):
    return pd.read_parquet(path, columns=columns)


## 2) Run pipeline stages
Skip any stage you don't need.

### 2.1 Build data

In [3]:
from pathlib import Path
import os

print("REPO_ROOT:", REPO_ROOT)
print("CONFIG_DIR:", CONFIG_DIR)
print("DATA_YAML:", DATA_YAML, "exists:", Path(DATA_YAML).exists())

# Also check module import from notebook kernel:
import sys
print("Python:", sys.executable)


REPO_ROOT: /Users/eshaanganguly/Documents/projects/algos 3 topstep
CONFIG_DIR: /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs
DATA_YAML: /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/data.yaml exists: True
Python: /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python


In [4]:
run_cmd([
    "python", "-m", CLI_MODULE, "build-data",
    "--config", str(DATA_YAML),
    "--run-id", RUN_ID,
    "--seed", str(SEED),
])


>> /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli build-data --config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/data.yaml --run-id baseline_v3_001 --seed 42
2025-12-23 15:53:18 [INFO] __main__: ================================================================================
2025-12-23 15:53:18 [INFO] __main__: V3 DATA PIPELINE - BUILD DATA
2025-12-23 15:53:18 [INFO] __main__: ================================================================================
2025-12-23 15:53:18 [INFO] __main__: Loaded config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/data.yaml
2025-12-23 15:53:18 [INFO] __main__: Run ID: baseline_v3_001
2025-12-23 15:53:18 [INFO] __main__: Output directory: runs/baseline_v3_001
2025-12-23 15:53:18 [INFO] __main__: Canonical bar size: 5m
2025-12-23 15:53:18 [INFO] __main__: Bar sizes to process: ['5m']
2025-12-23 15:53:18 [INFO] __main__: 
2025-12-2

CompletedProcess(args=['/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python', '-m', 'ml_intraday_v3.cli', 'build-data', '--config', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/data.yaml', '--run-id', 'baseline_v3_001', '--seed', '42'], returncode=0, stdout='', stderr="2025-12-23 15:53:18 [INFO] __main__: ================================================================================\n2025-12-23 15:53:18 [INFO] __main__: V3 DATA PIPELINE - BUILD DATA\n2025-12-23 15:53:18 [INFO] __main__: ================================================================================\n2025-12-23 15:53:18 [INFO] __main__: Loaded config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/data.yaml\n2025-12-23 15:53:18 [INFO] __main__: Run ID: baseline_v3_001\n2025-12-23 15:53:18 [INFO] __main__: Output directory: runs/baseline_v3_001\n2025-12-23 15:53:18 [INFO] __main__: Canonical bar size: 5m\n2025-12-23 15:53:18 [I

### 2.2 Build features

In [5]:
run_cmd([
    "python", "-m", CLI_MODULE, "build-features",
    "--run-dir", str(RUN_DIR),
    "--features-config", str(FEATURES_YAML),
])

>> /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli build-features --run-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001 --features-config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/features.yaml
2025-12-23 15:53:34 [INFO] __main__: ================================================================================
2025-12-23 15:53:34 [INFO] __main__: V3 FEATURE PIPELINE - BUILD FEATURES
2025-12-23 15:53:34 [INFO] __main__: ================================================================================
2025-12-23 15:53:34 [INFO] __main__: Run directory: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001
2025-12-23 15:53:34 [INFO] __main__: Loaded features config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/features.yaml
2025-12-23 15:53:34 [INFO] __main__: Found existing manifest with bar_sizes: ['5m']
2025-12-

CompletedProcess(args=['/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python', '-m', 'ml_intraday_v3.cli', 'build-features', '--run-dir', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001', '--features-config', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/features.yaml'], returncode=0, stdout='', stderr="2025-12-23 15:53:34 [INFO] __main__: ================================================================================\n2025-12-23 15:53:34 [INFO] __main__: V3 FEATURE PIPELINE - BUILD FEATURES\n2025-12-23 15:53:34 [INFO] __main__: ================================================================================\n2025-12-23 15:53:34 [INFO] __main__: Run directory: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001\n2025-12-23 15:53:34 [INFO] __main__: Loaded features config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/features.yaml\n2025-12-2

### 2.3 Build labels

In [6]:
run_cmd([
    "python", "-m", CLI_MODULE, "build-labels",
    "--run-dir", str(RUN_DIR),
    "--labeling-config", str(LABELING_YAML),
    "--execution-spec", str(EXECUTION_SPEC_YAML),
])

>> /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli build-labels --run-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001 --labeling-config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/labeling.yaml --execution-spec /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/execution_spec.yaml
2025-12-23 15:53:47 [INFO] __main__: ================================================================================
2025-12-23 15:53:47 [INFO] __main__: V3 LABEL PIPELINE - BUILD LABELS
2025-12-23 15:53:47 [INFO] __main__: ================================================================================
2025-12-23 15:53:47 [INFO] __main__: Run directory: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001
2025-12-23 15:53:47 [INFO] __main__: Loaded labeling config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/conf

CompletedProcess(args=['/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python', '-m', 'ml_intraday_v3.cli', 'build-labels', '--run-dir', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001', '--labeling-config', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/labeling.yaml', '--execution-spec', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/execution_spec.yaml'], returncode=0, stdout='', stderr="2025-12-23 15:53:47 [INFO] __main__: ================================================================================\n2025-12-23 15:53:47 [INFO] __main__: V3 LABEL PIPELINE - BUILD LABELS\n2025-12-23 15:53:47 [INFO] __main__: ================================================================================\n2025-12-23 15:53:47 [INFO] __main__: Run directory: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001\n2025-12-23 15:53:47 [INFO] __main__: Loaded labeling

### 2.4 Build weights

In [7]:
run_cmd([
    "python", "-m", CLI_MODULE, "build-weights",
    "--run-dir", str(RUN_DIR),
    "--labeling-config", str(LABELING_YAML),
])

>> /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli build-weights --run-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001 --labeling-config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/labeling.yaml
2025-12-23 15:53:50 [INFO] __main__: ================================================================================
2025-12-23 15:53:50 [INFO] __main__: V3 WEIGHT PIPELINE - BUILD WEIGHTS
2025-12-23 15:53:50 [INFO] __main__: ================================================================================
2025-12-23 15:53:50 [INFO] __main__: Run directory: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001
2025-12-23 15:53:50 [INFO] __main__: Loaded labeling config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/labeling.yaml
2025-12-23 15:53:50 [INFO] __main__: Found existing manifest with bar_sizes: ['5m']
2025-12-23 

CompletedProcess(args=['/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python', '-m', 'ml_intraday_v3.cli', 'build-weights', '--run-dir', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001', '--labeling-config', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/labeling.yaml'], returncode=0, stdout='', stderr="2025-12-23 15:53:50 [INFO] __main__: ================================================================================\n2025-12-23 15:53:50 [INFO] __main__: V3 WEIGHT PIPELINE - BUILD WEIGHTS\n2025-12-23 15:53:50 [INFO] __main__: ================================================================================\n2025-12-23 15:53:50 [INFO] __main__: Run directory: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001\n2025-12-23 15:53:50 [INFO] __main__: Loaded labeling config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/labeling.yaml\n2025-12-23 1

### 2.5 Build CV splits

In [8]:
run_cmd([
    "python", "-m", CLI_MODULE, "build-cv",
    "--run-dir", str(RUN_DIR),
    "--validation-config", str(VALIDATION_YAML),
])

>> /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli build-cv --run-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001 --validation-config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/validation.yaml
2025-12-23 15:53:51 [INFO] __main__: ================================================================================
2025-12-23 15:53:51 [INFO] __main__: V3 VALIDATION PIPELINE - BUILD CV
2025-12-23 15:53:51 [INFO] __main__: ================================================================================
2025-12-23 15:53:51 [INFO] __main__: Run directory: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001
2025-12-23 15:53:51 [INFO] __main__: Loaded validation config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/validation.yaml
2025-12-23 15:53:51 [INFO] __main__: Found existing manifest with bar_sizes: ['5m']
2025-12-2

CompletedProcess(args=['/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python', '-m', 'ml_intraday_v3.cli', 'build-cv', '--run-dir', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001', '--validation-config', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/validation.yaml'], returncode=0, stdout='', stderr="2025-12-23 15:53:51 [INFO] __main__: ================================================================================\n2025-12-23 15:53:51 [INFO] __main__: V3 VALIDATION PIPELINE - BUILD CV\n2025-12-23 15:53:51 [INFO] __main__: ================================================================================\n2025-12-23 15:53:51 [INFO] __main__: Run directory: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001\n2025-12-23 15:53:51 [INFO] __main__: Loaded validation config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/validation.yaml\n2025-12-23

### 2.6 Train

In [9]:
run_cmd([
    "python", "-m", CLI_MODULE, "build-train",
    "--run-dir", str(RUN_DIR),
    "--training-config", str(TRAINING_YAML),
    "--cv-kind", CV_KIND,
])

>> /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli build-train --run-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001 --training-config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/training.yaml --cv-kind purged_kfold
2025-12-23 15:53:52 [INFO] __main__: ================================================================================
2025-12-23 15:53:52 [INFO] __main__: V3 TRAINING PIPELINE - BUILD TRAIN
2025-12-23 15:53:52 [INFO] __main__: ================================================================================
2025-12-23 15:53:52 [INFO] __main__: Run directory: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001
2025-12-23 15:53:52 [INFO] __main__: Loaded training config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/training.yaml
2025-12-23 15:53:52 [INFO] __main__: Found existing manifest with bar_size

CompletedProcess(args=['/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python', '-m', 'ml_intraday_v3.cli', 'build-train', '--run-dir', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001', '--training-config', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/training.yaml', '--cv-kind', 'purged_kfold'], returncode=0, stdout='', stderr="2025-12-23 15:53:52 [INFO] __main__: ================================================================================\n2025-12-23 15:53:52 [INFO] __main__: V3 TRAINING PIPELINE - BUILD TRAIN\n2025-12-23 15:53:52 [INFO] __main__: ================================================================================\n2025-12-23 15:53:52 [INFO] __main__: Run directory: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001\n2025-12-23 15:53:52 [INFO] __main__: Loaded training config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/

### 2.7 Backtest

In [10]:
run_cmd([
    "python", "-m", CLI_MODULE, "build-backtest",
    "--run-dir", str(RUN_DIR),
    "--training-dir", str(RUN_DIR),
    "--backtest-config", str(BACKTEST_YAML),
    "--execution-spec", str(EXECUTION_SPEC_YAML),
    "--risk-config", str(RISK_YAML),
    "--cv-kind", CV_KIND,
])

>> /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli build-backtest --run-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001 --training-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001 --backtest-config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/backtest.yaml --execution-spec /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/execution_spec.yaml --risk-config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/risk.yaml --cv-kind purged_kfold
2025-12-23 15:54:09 [INFO] __main__: ================================================================================
2025-12-23 15:54:09 [INFO] __main__: V3 BACKTEST PIPELINE - BUILD BACKTEST
2025-12-23 15:54:09 [INFO] __main__: ================================================================================
2025-12-23 15:54:09 [INFO] __main__: Loaded 

CompletedProcess(args=['/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python', '-m', 'ml_intraday_v3.cli', 'build-backtest', '--run-dir', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001', '--training-dir', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001', '--backtest-config', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/backtest.yaml', '--execution-spec', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/execution_spec.yaml', '--risk-config', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/risk.yaml', '--cv-kind', 'purged_kfold'], returncode=0, stdout='', stderr="2025-12-23 15:54:09 [INFO] __main__: ================================================================================\n2025-12-23 15:54:09 [INFO] __main__: V3 BACKTEST PIPELINE - BUILD BACKTEST\n2025-12-23 15:54:09 [INFO] __main__: =====================

### 2.8 Experiments (optional)

In [11]:
run_cmd([
    "python", "-m", CLI_MODULE, "run-experiments",
    "--run-dir", str(RUN_DIR),
    "--grid-config", str(EXPERIMENT_GRID_YAML),
], check=False)

>> /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli run-experiments --run-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001 --grid-config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/experiment_grid.yaml
2025-12-23 15:54:13 [INFO] __main__: ================================================================================
2025-12-23 15:54:13 [INFO] __main__: V3 EXPERIMENT RUNNER
2025-12-23 15:54:13 [INFO] __main__: ================================================================================
2025-12-23 15:54:13 [WARNING] backtesting_v3.decisions: Missing primary predictions for 324 events; marking as skipped
2025-12-23 15:54:14 [WARNING] backtesting_v3.decisions: Missing primary predictions for 118 events; marking as skipped
2025-12-23 15:54:15 [WARNING] backtesting_v3.decisions: Missing primary predictions for 324 events; marking as skipped
2025-12-23 15:54:16 [WARNING] back

CompletedProcess(args=['/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python', '-m', 'ml_intraday_v3.cli', 'run-experiments', '--run-dir', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001', '--grid-config', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/experiment_grid.yaml'], returncode=0, stdout='', stderr='2025-12-23 15:54:13 [INFO] __main__: ================================================================================\n2025-12-23 15:54:13 [INFO] __main__: V3 EXPERIMENT RUNNER\n2025-12-23 15:54:13 [INFO] __main__: ================================================================================\n2025-12-23 15:54:13 [WARNING] backtesting_v3.decisions: Missing primary predictions for 324 events; marking as skipped\n2025-12-23 15:54:14 [WARNING] backtesting_v3.decisions: Missing primary predictions for 118 events; marking as skipped\n2025-12-23 15:54:15 [WARNING] backtesting_v3.decisions: Missing primar

### 2.9 Audit (optional)

In [12]:
run_cmd([
    "python", "-m", CLI_MODULE, "run-audit",
    "--run-dir", str(RUN_DIR),
    "--strict", "true",
], check=False)

>> /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli run-audit --run-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001 --strict true
2025-12-23 15:54:17 [INFO] __main__: ================================================================================
2025-12-23 15:54:17 [INFO] __main__: V3 AUDIT HARNESS
2025-12-23 15:54:17 [INFO] __main__: ================================================================================
Traceback (most recent call last):
  File "/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/lib/python3.11/site-packages/pandas/core/arrays/datetimelike.py", line 559, in _validate_comparison_value
    self._check_compatible_with(other)
  File "/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/lib/python3.11/site-packages/pandas/core/arrays/datetimes.py", line 542, in _check_compatible_with
    self._assert_tzawareness_compat(other)
  File "/Users/eshaanganguly/Documents/

CompletedProcess(args=['/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python', '-m', 'ml_intraday_v3.cli', 'run-audit', '--run-dir', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001', '--strict', 'true'], returncode=1, stdout='', stderr='2025-12-23 15:54:17 [INFO] __main__: ================================================================================\n2025-12-23 15:54:17 [INFO] __main__: V3 AUDIT HARNESS\n2025-12-23 15:54:17 [INFO] __main__: ================================================================================\nTraceback (most recent call last):\n  File "/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/lib/python3.11/site-packages/pandas/core/arrays/datetimelike.py", line 559, in _validate_comparison_value\n    self._check_compatible_with(other)\n  File "/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/lib/python3.11/site-packages/pandas/core/arrays/datetimes.py", line 542, in _check_compatible_wit

### 2.10 Walk-forward (optional)

In [13]:
run_cmd([
    "python", "-m", CLI_MODULE, "run-walkforward",
    "--run-dir", str(RUN_DIR),
    "--walkforward-config", str(WALKFORWARD_YAML),
], check=False)

>> /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli run-walkforward --run-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001 --walkforward-config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/walkforward.yaml
2025-12-23 15:54:19 [INFO] __main__: ================================================================================
2025-12-23 15:54:19 [INFO] __main__: V3 WALK-FORWARD EVALUATION
2025-12-23 15:54:19 [INFO] __main__: ================================================================================
2025-12-23 15:54:19 [INFO] training.dataset: Filtering 442 events where usable_for_training is False
/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value an

CompletedProcess(args=['/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python', '-m', 'ml_intraday_v3.cli', 'run-walkforward', '--run-dir', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/baseline_v3_001', '--walkforward-config', '/Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/walkforward.yaml'], returncode=0, stdout='', stderr="2025-12-23 15:54:19 [INFO] __main__: ================================================================================\n2025-12-23 15:54:19 [INFO] __main__: V3 WALK-FORWARD EVALUATION\n2025-12-23 15:54:19 [INFO] __main__: ================================================================================\n2025-12-23 15:54:19 [INFO] training.dataset: Filtering 442 events where usable_for_training is False\n/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be r

## 3) Inspect artifacts

In [14]:
print("Run directory contents:")
show_dir(RUN_DIR)

manifest_path = RUN_DIR / "run_manifest.json"
if manifest_path.exists():
    manifest = read_json(manifest_path)
    print("\nManifest top-level keys:")
    print(sorted(list(manifest.keys())))
else:
    print("No run_manifest.json found yet.")


Run directory contents:
bar_size=5m/backtests/purged_kfold/backtest_schema.json
bar_size=5m/backtests/purged_kfold/fold_0/backtest_metrics.json
bar_size=5m/backtests/purged_kfold/fold_0/equity.parquet
bar_size=5m/backtests/purged_kfold/fold_0/trades.parquet
bar_size=5m/backtests/purged_kfold/fold_1/backtest_metrics.json
bar_size=5m/backtests/purged_kfold/fold_1/equity.parquet
bar_size=5m/backtests/purged_kfold/fold_1/trades.parquet
bar_size=5m/backtests/purged_kfold/fold_2/backtest_metrics.json
bar_size=5m/backtests/purged_kfold/fold_2/equity.parquet
bar_size=5m/backtests/purged_kfold/fold_2/trades.parquet
bar_size=5m/backtests/purged_kfold/fold_3/backtest_metrics.json
bar_size=5m/backtests/purged_kfold/fold_3/equity.parquet
bar_size=5m/backtests/purged_kfold/fold_3/trades.parquet
bar_size=5m/backtests/purged_kfold/fold_4/backtest_metrics.json
bar_size=5m/backtests/purged_kfold/fold_4/equity.parquet
bar_size=5m/backtests/purged_kfold/fold_4/trades.parquet
bar_size=5m/backtests/purged_k

### 3.1 Quick per-bar inspections

In [15]:
def inspect_bar(bar_size: str):
    p = RUN_DIR / f"bar_size={bar_size}"
    print("\n" + "="*80)
    print("BAR SIZE:", bar_size)
    print("="*80)

    for name in ["bars.parquet","features.parquet","events.parquet","weights.parquet","qa_report.json"]:
        f = p / name
        print(f"{name}: {'OK' if f.exists() else 'MISSING'}")

    if (p/"bars.parquet").exists():
        bars = read_parquet(p/"bars.parquet")
        print("bars:", bars.shape, "index:", bars.index.min(), "->", bars.index.max())
        if "is_synthetic" in bars.columns:
            print("synthetic bars:", int(bars["is_synthetic"].sum()))

    if (p/"features.parquet").exists():
        feats = read_parquet(p/"features.parquet")
        print("features:", feats.shape)
        if "usable_for_training" in feats.columns:
            print("usable:", int(feats["usable_for_training"].sum()), "/", len(feats))

    if (p/"events.parquet").exists():
        events = read_parquet(p/"events.parquet")
        print("events:", events.shape)
        if "y" in events.columns:
            print("y counts:", events["y"].value_counts(dropna=False).to_dict())

    if (p/"weights.parquet").exists():
        w = read_parquet(p/"weights.parquet")
        print("weights:", w.shape)
        if "w_final" in w.columns:
            print("w_final describe:\n", w["w_final"].describe())

for bs in BAR_SIZES:
    inspect_bar(bs)



BAR SIZE: 1m
bars.parquet: MISSING
features.parquet: MISSING
events.parquet: MISSING
weights.parquet: MISSING
qa_report.json: MISSING

BAR SIZE: 5m
bars.parquet: OK
features.parquet: OK
events.parquet: OK
weights.parquet: OK
qa_report.json: OK
bars: (127452, 13) index: 2019-05-06 13:30:00+00:00 -> 2025-12-02 20:55:00+00:00
synthetic bars: 16
features: (127452, 35)
usable: 126594 / 127452
events: (63496, 17)
y counts: {0.0: 33663, -1.0: 15416, 1.0: 14417}
weights: (63496, 6)
w_final describe:
 count    63496.000000
mean         0.105235
std          0.024193
min          0.050563
25%          0.088211
50%          0.101798
75%          0.117970
max          0.425048
Name: w_final, dtype: float64


## 4) Run tests (optional)

In [16]:
# Run a few core suites (edit as needed)
run_cmd(["pytest", "ml_intraday_v3/tests/test_labels.py", "-q"], check=False)
run_cmd(["pytest", "ml_intraday_v3/tests/test_backtest.py", "-q"], check=False)
run_cmd(["pytest", "ml_intraday_v3/tests/test_audit.py", "-q"], check=False)


>> /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m pytest ml_intraday_v3/tests/test_labels.py -q
.......                                                                  [100%]
7 passed in 1.07s

>> /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m pytest ml_intraday_v3/tests/test_backtest.py -q
.......                                                                  [100%]
7 passed in 1.07s

>> /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m pytest ml_intraday_v3/tests/test_audit.py -q
....                                                                     [100%]
4 passed in 1.08s



CompletedProcess(args=['/Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python', '-m', 'pytest', 'ml_intraday_v3/tests/test_audit.py', '-q'], returncode=0, stdout='\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b                                                                     [100%]\x1b\n\x1b\x1b\x1b4 passed\x1b\x1b in 1.08s\x1b\x1b\n', stderr='')